In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [8]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is exploring fictional information regarding the lunar city "Lunapolis," specifically its environment, population, and current political climate.\n\n## SUMMARY\n*   **Location:** The capital of the moon is Lunapolis.\n*   **Environment:** Lunapolis experiences extreme temperatures, ranging from -100C to 120C, with clear skies.\n*   **Demographics:** The city has a population of 100,000 "cheese miners."\n*   **Political Context:** The cheese miners\' union is expected to strike due to dissatisfaction with the city\'s new president.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nAwait further user inquiries regarding the situation in Lunapolis or additional fictional lunar data.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='43fffd53-52d3-43bc-bb11-c58e2b1695f8'),
              HumanMessage(content="If you were Lunapolis' new president how would you r

In [9]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is exploring fictional information regarding the lunar city "Lunapolis," specifically its environment, population, and current political climate.

## SUMMARY
*   **Location:** The capital of the moon is Lunapolis.
*   **Environment:** Lunapolis experiences extreme temperatures, ranging from -100C to 120C, with clear skies.
*   **Demographics:** The city has a population of 100,000 "cheese miners."
*   **Political Context:** The cheese miners' union is expected to strike due to dissatisfaction with the city's new president.

## ARTIFACTS
None.

## NEXT STEPS
Await further user inquiries regarding the situation in Lunapolis or additional fictional lunar data.


## Trim/delete messages

In [10]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

In [ ]:
print(response["messages"][-1].content)